#### 0. Import libraries

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package to navigate through files and folders
import os

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
import opentnsim.fis as fis
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.plotutils import generate_vessel_gantt_chart

# import of modules important for locking
from opentnsim.lock import lock as lock_module
from opentnsim.vessel_traffic_service import vessel_traffic_service as vessel_traffic_service_module

# energy module packages added from 0103 notebook
from opentnsim.energy.logutils import (
    add_energy_attributes_to_eventtable,
    add_fuel_attributes_to_event_table,
    add_H2_attributes_to_event_table)
from opentnsim.graph import mixins as graph_module

# package(s) needed for inspecting the output
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# package(s) needed for inspecting the output
import pandas as pd
import geopandas as gpd
import numpy as np

# plot libraries
import folium
import pickle

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 2.2.2.dev47+gbe28de4fe.d20260122


#### 

In [6]:
sub_FG = lock_information['modified_graph']
lock_length = lock_information["lock_length"]
lock_depth = lock_information["lock_depth"]
node_A = lock_information["registration_node_A"]
node_B = lock_information["registration_node_B"]

KeyError: 'modified_graph'

In [ ]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        opentnsim.energy.mixins.ConsumesEnergy,
        lock_module.PassesLockComplex,             # allows to interact with a lock
        opentnsim.core.Identifiable,               # allows to give the object a name and a random ID,
        opentnsim.core.Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        opentnsim.core.VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        opentnsim.core.ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        opentnsim.mixins.HasMultiDiGraph,           # allow to operate on a graph that can include parallel edges from and to the same nodes
        opentnsim.output.HasOutput,                # allow additional output to be stored
    ), 
    {}
)

#### 1. Run simulation
##### 1.1 Set mission and simpy environment with VTS

In [ ]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [ ]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

# This is not the best way to assign the lock_depth but it works 
# Add constant general depth
for u, v in sub_FG.edges():
    # FG.edges[u, v]['GeneralDepth'] = 6
    sub_FG.edges[u, v]['GeneralDepth'] = lock_depth
    
env.graph = sub_FG

# add components important for locking to the environment
env.vessel_traffic_service = vessel_traffic_service_module.VesselTrafficService(graph=sub_FG, crs_m = "EPSG:28992") #Amersfoort / RD New reference system

##### 3.2 Create lock object

In [ ]:
waiting_time_before_lock = 840 #seconds

lock = lock_module.IsLockComplex(
    env = env,
    name = 'Lock',
    node_open = lock_information["lock_node_A"],
    node_A = lock_information["lock_node_A"],
    node_B = lock_information["lock_node_B"],
    edge_waiting_area_A = lock_information["edge_waiting_area_A"], 
    edge_waiting_area_B = lock_information["edge_waiting_area_B"],
    distance_lock_doors_A_to_waiting_area_A = 500.,
    distance_lock_doors_B_to_waiting_area_B = 500.,
    distance_from_start_node_to_lock_doors_A = lock_information["distance_from_start_node_to_lock_doors_A"],
    distance_from_end_node_to_lock_doors_B = lock_information["distance_from_end_node_to_lock_doors_B"],
    speed_reduction_length_before_waiting_area = 600,
    sailing_out_speed_A = 1.5,
    sailing_out_speed_B = 1.5,
    speed_reduction_factor = 0.5,
    speed_reduction_factor_waiting_area_A = 0.5,
    speed_reduction_factor_waiting_area_B = 0.5,
    lock_length = lock_information["lock_length"],
    lock_width = lock_information["lock_width"],
    lock_depth = lock_information["lock_depth"],
    mandatory_waiting_time_before_lock = waiting_time_before_lock,
    levelling_time = 550,
    sailing_distance_to_crossing_point = 300,
    doors_opening_time= 100,
    doors_closing_time= 100,
    sailing_in_speed_sea = 1.0,
    sailing_in_speed_canal = 1.0,
    registration_nodes = [lock_information["registration_node_A"], lock_information["registration_node_B"]],
)

In [ ]:
# print(lock_information["lock_depth"])
# print(lock.lock_depth)

##### 3.3 Create vessels

In [ ]:
# create vessels from dict 
data_vessel_1 = {
    "env": env,
    "name": 'Vessel',           # you can give the vessel an arbitratry name
    "type": 'Va/M9 - Verl. Groot Rijnschip', # This indicates the vessel class. This info is mainly informative.
    "L": 110,                   # m
    "B": 11.4,                  # m
    "T": 3.5,                   # m
    "v": 4,                     # m/s If None: this value is calculated based on P_tot_given
    "safety_margin": 0.2,       # for tanker vessel with sandy bed the safety margin is recommended as 0.2 m 
    "h_squat": False,            # if the ship should squat while moving, set to True, otherwise set to False
    "P_installed": 1750.0,      # kW
    "P_tot_given": None,        # kW If None: this value is calculated value based on speed
    "bulbous_bow": False,       # if a vessel has no bulbous_bow, set to False; otherwise set to True.
    "karpov_correction": False, # if False, don't apply the karpov correction, if True, apply the karpov correction
    "P_hotel_perc": 0.05,       # 0: all power goes to propulsion
    "P_hotel": None,            # None: calculate P_hotel from percentage
    "x": 2,                     # number of propellers
    "L_w": 3.0 ,
    "C_B": 0.85,                # block coefficient 
    "C_year": 1990,             # engine build year
    "arrival_time": datetime.datetime(2025, 1, 1, 0, 0, 0),
    # "geometry": env.graph.nodes[route[0]]['geometry'],
    # "route": route,             # the route to sail
    "geometry": env.graph.nodes[node_A]['geometry'],        # required by Locatable
    "route": nx.dijkstra_path(env.graph, node_A, node_B),      # required by Routeable
}  #  
vessel_1 = Vessel(**data_vessel_1)
vessel_1.name = 'Vessel 1'

# start the simulation
env.process(mission(env, vessel_1))
env.run()

In [ ]:
trajectory_gdf = pickle.load(open(os.path.join(path, "..", "data", "volkerak_track.pickle"),'rb'))

In [ ]:
# Het lukt me even niet om de modelled en AIS tegelijk te plotten
# Het lijkt alsof er iets is veranderd in def lock.create_time_distance_plot

xlimmin = -2000
xlimmax = 2000
ylimmin = pd.Timestamp('2025-09-02 9:50:00')
ylimmax = pd.Timestamp('2025-09-02 11:00:00')

# We can plot the time-distance diagram
fig,ax = plt.subplots()

lock.create_time_distance_plot([vessel_1], xlimmin = xlimmin, xlimmax = xlimmax, ax = ax, label='modelled')

plt.show()

# ax.plot(trajectory_gdf.distance_cumulative,trajectory_gdf.index, label='AIS')
# ax.set_xlim([xlimmin,xlimmax])
# ax.fill([-lock_length/2,-lock_length/2,lock_length/2,lock_length/2],
#         [ylimmin,ylimmax,ylimmax,ylimmin],color='lightgrey',zorder=-1)
# ax.set_ylim([ylimmin,ylimmax])
# ax.legend()
# plt.show()

In [ ]:
xlimmin = -2000
xlimmax = 2000
ylimmin = pd.Timestamp('2025-09-02 9:50:00')
ylimmax = pd.Timestamp('2025-09-02 11:00:00')

# We can plot the time-distance diagram
fig,ax = plt.subplots()

ax.plot(trajectory_gdf.distance_cumulative,trajectory_gdf.index, label='AIS', color='orange')
ax.set_xlim([xlimmin,xlimmax])
ax.fill([-lock_length/2,-lock_length/2,lock_length/2,lock_length/2],
        [ylimmin,ylimmax,ylimmax,ylimmin],color='lightgrey',zorder=-1)
ax.set_ylim([ylimmin,ylimmax])
ax.legend()

plt.show()

In [ ]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel_1.logbook)

print("'{}' logbook data:".format(vessel_1.name))  
print('')

display(df)

In [ ]:
lock.operation_planning.iloc[0]

In [ ]:
lock.vessel_planning.iloc[0]

In [ ]:
df_eventtable = logbook2eventtable([vessel_1])
# df_eventtable.head(5)
df_eventtable
# print("'{}' logbook data:".format(vessel_1.name))  

In [ ]:
generate_vessel_gantt_chart(df_eventtable)

In [ ]:
# add energy attributes to the minimum event table
lock_depth = lock_information["lock_depth"]
df_eventtable_energy = add_energy_attributes_to_eventtable(df_eventtable, [vessel_1])        

# pd.set_option('display.max_columns', None)
df_eventtable_energy

In [ ]:
df_eventtable_energy_H2 = add_H2_attributes_to_event_table(df_eventtable, [vessel_1])

pd.set_option('display.max_columns', None)
df_eventtable_energy_H2

In [ ]:
# Event totals + unit conversion + speed 
df_event = (
    df_eventtable
    .groupby('object name')
    .agg({'duration (s)': 'sum', 'distance (m)': 'sum'})
)
# df_event['duration (h)']      = df_event['duration (s)'] / 3600
df_event['duration (min)']      = df_event['duration (s)'] / 60
df_event['distance (km)']     = df_event['distance (m)'] / 1000
df_event['avg_speed_m_s']     = df_event['distance (m)'] / df_event['duration (s)']
# df_event['avg_speed_km_h']    = df_event['avg_speed_m_s'] * 3.6

# H2 energy & fuel
df_H2 = (
    df_eventtable_energy_H2
    .groupby('object name')
    .agg({'total_energy (kWh)': 'sum', 'H2_consumption (g)': 'sum'})
)
df_H2['H2_consumption (kg)'] = df_H2['H2_consumption (g)'] / 1000

# --- Final summary table ---
summary = (
    df_event
    .join(df_H2[['H2_consumption (kg)']])
)

summary